In [1]:
import pandas as pd

df = pd.read_csv("shipment_master.csv")

print(df.columns.tolist())
print(df.head())

['shipment_id', 'shipment_date', 'carrier', 'weight_kg', 'distance_km', 'freight_cost', 'planned_delivery_date', 'actual_delivery_date', 'otd']
  shipment_id shipment_date    carrier  weight_kg  distance_km  freight_cost  \
0    SHP00001    2025-05-06  Carrier_A       2303          152        698.64   
1    SHP00002    2025-10-30  Carrier_A        762         2283       2374.56   
2    SHP00003    2025-04-30  Carrier_A       1841          433        857.52   
3    SHP00004    2025-11-29  Carrier_A       1678         2348       2656.80   
4    SHP00005    2025-05-23  Carrier_B       4877         1889       2486.60   

  planned_delivery_date actual_delivery_date  otd  
0            2025-05-10           2025-05-10    1  
1            2025-11-06           2025-11-06    1  
2            2025-05-09           2025-05-09    1  
3            2025-12-08           2025-12-08    1  
4            2025-05-24           2025-05-26    0  


In [2]:
# ==========================================
# LOGISTICS ANALYTICS PROJECT
# ==========================================

import pandas as pd
import numpy as np

# ------------------------------------------
# LOAD DATA
# ------------------------------------------

df = pd.read_csv("shipment_master.csv")

print("Dataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
print(df.head())

# ------------------------------------------
# DATE CONVERSION
# ------------------------------------------

df["shipment_date"] = pd.to_datetime(df["shipment_date"])
df["planned_delivery_date"] = pd.to_datetime(df["planned_delivery_date"])
df["actual_delivery_date"] = pd.to_datetime(df["actual_delivery_date"])

# ------------------------------------------
# DELIVERY LEAD TIME
# ------------------------------------------

df["delivery_days"] = (
    df["actual_delivery_date"]
    - df["shipment_date"]
).dt.days

# ------------------------------------------
# COST PER KM
# ------------------------------------------

df["cost_per_km"] = (
    df["freight_cost"]
    / df["distance_km"]
)

# ------------------------------------------
# COST PER KG
# ------------------------------------------

df["cost_per_kg"] = (
    df["freight_cost"]
    / df["weight_kg"]
)

# ------------------------------------------
# OVERALL OTD %
# ------------------------------------------

overall_otd = (
    df["otd"].mean()
) * 100

print("\nOverall OTD %:")
print(round(overall_otd,2))

# ------------------------------------------
# TOTAL FREIGHT COST
# ------------------------------------------

total_freight_cost = (
    df["freight_cost"]
    .sum()
)

print("\nTotal Freight Cost:")
print(round(total_freight_cost,2))

# ------------------------------------------
# TOTAL SHIPMENTS
# ------------------------------------------

total_shipments = len(df)

print("\nTotal Shipments:")
print(total_shipments)

# ------------------------------------------
# CARRIER PERFORMANCE
# ------------------------------------------

carrier_performance = (
    df.groupby("carrier")
      .agg(
          shipments=("shipment_id","count"),
          freight_cost=("freight_cost","sum"),
          avg_otd=("otd","mean"),
          avg_delivery_days=("delivery_days","mean"),
          avg_cost_per_km=("cost_per_km","mean"),
          avg_cost_per_kg=("cost_per_kg","mean")
      )
      .reset_index()
)

carrier_performance["avg_otd"] = (
    carrier_performance["avg_otd"] * 100
)

carrier_performance = (
    carrier_performance
    .sort_values(
        "avg_otd",
        ascending=False
    )
)

print("\nCarrier Performance:")
print(carrier_performance)

# ------------------------------------------
# BEST CARRIER
# ------------------------------------------

best_carrier = (
    carrier_performance
    .sort_values(
        "avg_otd",
        ascending=False
    )
    .head(1)
)

print("\nBest Carrier:")
print(best_carrier)

# ------------------------------------------
# WORST CARRIER
# ------------------------------------------

worst_carrier = (
    carrier_performance
    .sort_values(
        "avg_otd",
        ascending=True
    )
    .head(1)
)

print("\nWorst Carrier:")
print(worst_carrier)

# ------------------------------------------
# MONTHLY FREIGHT COST
# ------------------------------------------

df["shipment_month"] = (
    df["shipment_date"]
    .dt.to_period("M")
)

monthly_cost = (
    df.groupby("shipment_month")
      .agg(
          freight_cost=("freight_cost","sum"),
          shipments=("shipment_id","count")
      )
      .reset_index()
)

print("\nMonthly Freight Cost:")
print(monthly_cost)

# ------------------------------------------
# MONTHLY OTD
# ------------------------------------------

monthly_otd = (
    df.groupby("shipment_month")
      .agg(
          otd_percentage=("otd","mean")
      )
      .reset_index()
)

monthly_otd["otd_percentage"] = (
    monthly_otd["otd_percentage"] * 100
)

print("\nMonthly OTD:")
print(monthly_otd)

# ------------------------------------------
# LONGEST DISTANCE SHIPMENTS
# ------------------------------------------

long_distance = (
    df.sort_values(
        "distance_km",
        ascending=False
    )
    .head(10)
)

print("\nLongest Distance Shipments:")
print(
    long_distance[
        [
            "shipment_id",
            "carrier",
            "distance_km",
            "freight_cost"
        ]
    ]
)

# ------------------------------------------
# MOST EXPENSIVE SHIPMENTS
# ------------------------------------------

expensive_shipments = (
    df.sort_values(
        "freight_cost",
        ascending=False
    )
    .head(10)
)

print("\nMost Expensive Shipments:")
print(
    expensive_shipments[
        [
            "shipment_id",
            "carrier",
            "freight_cost",
            "distance_km"
        ]
    ]
)

# ------------------------------------------
# EXPORT FILES FOR POWER BI
# ------------------------------------------

carrier_performance.to_csv(
    "carrier_performance.csv",
    index=False
)

monthly_cost.to_csv(
    "monthly_freight_cost.csv",
    index=False
)

monthly_otd.to_csv(
    "monthly_otd.csv",
    index=False
)

print("\n===================================")
print("Logistics Analysis Complete!")
print("Files exported successfully.")
print("===================================")

Dataset Shape:
(2000, 9)

Columns:
['shipment_id', 'shipment_date', 'carrier', 'weight_kg', 'distance_km', 'freight_cost', 'planned_delivery_date', 'actual_delivery_date', 'otd']

First 5 Rows:
  shipment_id shipment_date    carrier  weight_kg  distance_km  freight_cost  \
0    SHP00001    2025-05-06  Carrier_A       2303          152        698.64   
1    SHP00002    2025-10-30  Carrier_A        762         2283       2374.56   
2    SHP00003    2025-04-30  Carrier_A       1841          433        857.52   
3    SHP00004    2025-11-29  Carrier_A       1678         2348       2656.80   
4    SHP00005    2025-05-23  Carrier_B       4877         1889       2486.60   

  planned_delivery_date actual_delivery_date  otd  
0            2025-05-10           2025-05-10    1  
1            2025-11-06           2025-11-06    1  
2            2025-05-09           2025-05-09    1  
3            2025-12-08           2025-12-08    1  
4            2025-05-24           2025-05-26    0  

Overall OTD 